## Load data

In [1]:
import pandas as pd

daily = pd.read_csv("salesdaily.csv")
daily["datum"] = pd.to_datetime(daily["datum"], format="%m/%d/%Y")

ATC_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

daily.head()

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,Year,Month,Hour,Weekday Name
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0,2014,1,276,Sunday
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0,2014,1,276,Monday


## 1. Total sales quantity for each drug category

In [2]:
def total_sales_by_category(df):
    return df[ATC_COLS].sum().sort_values(ascending=False)

total_sales_by_category(daily)

N02BE    63005.402708
N05B     18645.737500
R03      11608.822917
M01AB    10600.937083
M01AE     8204.618646
N02BA     8172.209000
R06       6107.817500
N05C      1249.958333
dtype: float64

## 2. Highest-selling individual "drug brand"

(no brand column exists in the data -> same result as the category totals above)

In [3]:
def highest_selling_category(df):
    totals = total_sales_by_category(df)
    return totals.index[0], totals.iloc[0]

cat, qty = highest_selling_category(daily)
print(f"{cat}: {qty:.2f} total units")

N02BE: 63005.40 total units


## 3. Top 3 categories for specific year/month combinations

In [4]:
def top3_for_month(df, year, month):
    mask = (df["Year"] == year) & (df["Month"] == month)
    return df.loc[mask, ATC_COLS].sum().sort_values(ascending=False).head(3)

for year, month, label in [(2015, 1, "January 2015"),
                            (2016, 7, "July 2016"),
                            (2017, 9, "September 2017")]:
    print(f"--- {label} ---")
    print(top3_for_month(daily, year, month), "\n")

--- January 2015 ---
N02BE    1044.24
N05B      463.00
R03       177.25
dtype: float64 

--- July 2016 ---
N02BE    652.362000
N05B     240.000000
M01AB    194.528333
dtype: float64 

--- September 2017 ---
N02BE    863.75
N05B     223.00
R03      139.00
dtype: float64 



## 4. Category sold most often in 2017

Two readings of "most often":
- by total **quantity**
- by **frequency** (number of days with any nonzero sales)

In [5]:
def most_sold_in_year(df, year):
    sub = df[df["Year"] == year]
    by_quantity = sub[ATC_COLS].sum().sort_values(ascending=False)
    by_frequency = (sub[ATC_COLS] > 0).sum().sort_values(ascending=False)
    return by_quantity, by_frequency

by_qty, by_freq = most_sold_in_year(daily, 2017)
print("By total quantity:")
print(by_qty, "\n")
print("By frequency (days with nonzero sales):")
print(by_freq)

By total quantity:
N02BE    9258.804833
N05B     2555.541667
R03      1893.614583
M01AB    1846.617083
M01AE    1387.298333
N02BA    1288.295833
R06       988.860000
N05C      180.333333
dtype: float64 

By frequency (days with nonzero sales):
N02BE    361
M01AB    360
M01AE    360
N05B     356
N02BA    350
R06      329
R03      286
N05C      99
dtype: int64


## 5. Category with the highest average daily sales

In [6]:
def highest_average_daily_sales(df):
    return df[ATC_COLS].mean().sort_values(ascending=False)

highest_average_daily_sales(daily)

N02BE    29.917095
N05B      8.853627
R03       5.512262
M01AB     5.033683
M01AE     3.895830
N02BA     3.880441
R06       2.900198
N05C      0.593522
dtype: float64

## 6. Seasonality check for R03 (respiratory drugs)

In [7]:
def r03_seasonality(df):
    month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    result = df.groupby("Month")["R03"].agg(["sum", "mean"]).round(2)
    result.index = month_names
    return result.sort_values("sum", ascending=False)

r03_seasonality(daily)

,sum,mean
Jan,1264.66,6.84
Dec,1227.00,7.92
Oct,1175.00,7.21
Mar,1170.00,6.29
Feb,1165.58,6.90
Apr,1038.88,5.77
Nov,934.00,6.23
May,931.29,5.01
Sep,792.42,4.40
Jun,783.00,4.35
